# 기존 MLP의 작은 고정 배치 학습 점검

현재 `main.ipynb`에서 확인한 데이터 준비·모델 초기화·순전파·학습/평가 함수의 코드를 변경 없이 옮긴 시작 자료입니다. 원본의 코드·저장 출력은 보존했고 이 사본의 출력과 실행 번호만 비웠습니다. 기존 AI의 데이터·모델 준비 및 API 도움 이력을 유지합니다.

`Python 3.14 (llm-learning-lab)` 커널을 새로 시작한 뒤 위에서부터 준비 셀을 실행합니다. 이 파일은 아직 실행하지 않았습니다. 초기화 셀은 학습 전 실행하며, 학습과 사후 평가 사이에 다시 실행하면 학습된 가중치가 사라집니다.

읽을 범위: Andrej Karpathy, [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/) (2019-04-25), §2의 **overfit one batch** 항목만 읽습니다. ROADMAP P0의 기존 MLP 진단 범위이며 새 강의나 무보조 재구현이 아닙니다.

목적은 소수의 같은 예제에 반복적으로 맞출 수 있는지 확인하는 것입니다. 학습 대상의 선택·코드 조정·실행·결과 해석은 학습자가 직접 합니다.

In [1]:
import random
from urllib.request import urlopen

import torch
import torch.nn.functional as F

with urlopen("https://raw.githubusercontent.com/karpathy/makemore/master/names.txt", timeout=30) as response:
    words = response.read().decode("utf-8").splitlines()

chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3
embedding_dim = 10
hidden_size = 200
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("words:", len(words), "vocabulary:", vocab_size, "device:", device)

words: 32033 vocabulary: 27 device: cuda


In [2]:
def build_dataset(word_list):
    X, Y = [], []
    for word in word_list:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return (
        torch.tensor(X, dtype=torch.long, device=device),
        torch.tensor(Y, dtype=torch.long, device=device),
    )

shuffled_words = words.copy()
random.Random(42).shuffle(shuffled_words)
n = len(shuffled_words)
train_end, dev_end = n * 8 // 10, n * 9 // 10
Xtr, Ytr = build_dataset(shuffled_words[:train_end])
Xdev, Ydev = build_dataset(shuffled_words[train_end:dev_end])
Xte, Yte = build_dataset(shuffled_words[dev_end:])
for split_name, split_X, split_Y in [("train", Xtr, Ytr), ("dev", Xdev, Ydev), ("test", Xte, Yte)]:
    print(split_name, split_X.shape, split_Y.shape)
del split_name, split_X, split_Y

train torch.Size([182625, 3]) torch.Size([182625])
dev torch.Size([22655, 3]) torch.Size([22655])
test torch.Size([22866, 3]) torch.Size([22866])


In [3]:
# 재실행하면 가중치와 난수 시퀀스를 처음 상태로 되돌립니다.
g_model = torch.Generator(device=device).manual_seed(2147483647)
g_batch = torch.Generator(device=device).manual_seed(42)
C = torch.randn((vocab_size, embedding_dim), generator=g_model, device=device, dtype=torch.float32)
W1 = torch.randn((block_size * embedding_dim, hidden_size), generator=g_model, device=device, dtype=torch.float32)
b1 = torch.randn(hidden_size, generator=g_model, device=device, dtype=torch.float32)
W2 = torch.randn((hidden_size, vocab_size), generator=g_model, device=device, dtype=torch.float32) * 0.02
b2 = torch.zeros(vocab_size, device=device, dtype=torch.float32)
W_direct = torch.zeros((block_size * embedding_dim, vocab_size), dtype=W1.dtype, device=W1.device)
parameters_direct = [C, W1, b1, W2, b2, W_direct]
for p in parameters_direct:
    p.requires_grad_(True)

print("parameters:", sum(p.numel() for p in parameters_direct))


parameters: 12707


In [4]:
def forward_direct(X):
    emb = C[X]
    flat = emb.flatten(start_dim=1)
    direct_scores = flat @ W_direct
    h = torch.tanh(flat @ W1 + b1)
    return h @ W2 + b2 + direct_scores

## 기존 학습·평가 함수

아래 함수는 원본 그대로입니다. 현재 `train_step_direct`는 반복마다 전체 train에서 새 인덱스를 추출합니다. 이번 실습에서는 직접 조정해 반복마다 동일한 고정 배치 전체를 학습하도록 만듭니다. 수정한 함수 셀은 다시 실행해야 새 정의가 적용됩니다. `eval_direct`는 동일 배치의 학습 전후 평가에 사용할 수 있습니다.

In [5]:
def train_step_direct(iter, lr):
    for step in range(iter):
        indices = torch.randint(high=len(Xtr), size=(batch_size,), generator=g_batch, device="cuda")
        loss = F.cross_entropy(forward_direct(Xtr[indices]), Ytr[indices])

        for p in parameters_direct:
            p.grad = None

        loss.backward()

        with torch.no_grad():
            for p in parameters_direct:
                p -= p.grad * lr

@torch.no_grad()
def eval_direct(X, Y):
    return F.cross_entropy(forward_direct(X), Y)

## 이번 시도의 조건과 저장할 결과

- train에서 문맥이 서로 다른 예제 여덟 개를 골라 한 배치로 고정합니다. 입력과 정답은 같은 원래 행에서 함께 선택합니다. 선택한 입력·정답과 각각의 shape를 출력합니다.
- 같은 문맥에 서로 다른 정답이 붙으면 두 정답을 동시에 확률 1로 예측할 수 없습니다. 이번 진단에서는 문맥이 다른 예제를 사용해 이 충돌을 피합니다.
- 학습 전 고정 배치의 손실을 저장합니다. 기존 모델 구조·초기화·학습률을 유지하고, 첫 시도는 기존 `main.ipynb`와 같은 반복 횟수로 진행합니다.
- 반복 중 새 예제를 뽑지 않고, 선택한 여덟 개 전체로 매번 손실을 계산하도록 학습 함수를 직접 조정합니다.
- 학습 후 같은 배치의 손실과 각 예제의 예측 문자 ID·정답 ID를 출력합니다. 모든 실습 셀을 실행하고 저장한 뒤 결과를 한 문장으로 해석합니다.

살펴볼 결과는 손실이 0에 가까워지는지와 각 예측이 정답에 맞는지입니다. 정해진 반복 횟수 안의 달성을 보장하거나 공식 과제의 통과 기준으로 삼는 것은 아닙니다. 원하는 수준에 이르지 못하면 실제 코드·출력으로 원인을 살펴봅니다. 이번 배치 적합 결과만으로 새로운 데이터의 성능을 판단할 수는 없습니다.

아래 빈 셀은 차례로 배치 선택·학습 전 평가, 학습 호출, 학습 후 평가에 사용하고 마지막 빈 마크다운 셀에 해석을 적습니다.

In [6]:
indices = torch.randperm(len(Xtr))[:8]
print(f"indices: {indices}")

indices: tensor([ 89465, 143532, 106333,   8656, 101605, 169236,  92872,  10850])


In [ ]:
def train_step_direct_repeat(iter, lr):
    for step in range(iter):
        loss = F.cross_entropy(forward_direct(Xtr[indices]), Ytr[indices])

        for p in parameters_direct:
            p.grad = None

        loss.backward()

        with torch.no_grad():
            for p in parameters_direct:
                p -= p.grad * lr

loss_before = eval_direct(Xtr[indices], Ytr[indices])
train_step_direct_repeat(iter=1000, lr=0.03)
loss_after = eval_direct(Xtr[indices], Ytr[indices])

print(f"loss_before: {loss_before}")
print(f"loss_after: {loss_after}")

loss_before: 3.2520527839660645
loss_after: 0.0013941442593932152


In [10]:
print(f"Xtr[indices]: {Xtr[indices]}")
print(f"Ytr[indices]: {Ytr[indices]}")
predict = forward_direct(Xtr[indices])
ids = torch.argmax(predict, dim=1)
print(f"ids: {ids}")

Xtr[indices]: tensor([[ 0,  0,  8],
        [ 0,  0, 23],
        [11, 15,  2],
        [12, 12,  1],
        [ 1, 12, 25],
        [ 9, 11, 11],
        [ 8,  5,  1],
        [25, 15, 14]], device='cuda:0')
Ytr[indices]: tensor([ 9,  9,  9, 18, 12,  1, 12,  1], device='cuda:0')
ids: tensor([ 9,  9,  9, 18, 12,  1, 12,  1], device='cuda:0')
